<a href="https://colab.research.google.com/github/one-2730/ESSA-25-1/blob/Assignment/ESAA_OB_0310_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import warnings
warnings.filterwarnings('ignore')

# import package
import numpy as np
import os

#5장에서 소개한 moons dataset 불러오기
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
X,y = make_moons(n_samples=1000, noise=0.15)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

#7. 앙상블 학습과 랜덤 포레스트

##7.1 투표 기반 분류기

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

log_clf = LogisticRegression()
rnd_clf = RandomForestClassifier()
svm_clf = SVC()

voting_clf = VotingClassifier(
    estimators = [('lr', log_clf), ('rf', rnd_clf), ('svc', svm_clf)],
    voting = 'hard'
)
voting_clf.fit(X_train, y_train)

VotingClassifier(estimators=[('lr', LogisticRegression()),
                             ('rf', RandomForestClassifier()), ('svc', SVC())])

In [8]:
from sklearn.metrics import accuracy_score
for clf in (log_clf, rnd_clf, svm_clf, voting_clf):
  clf.fit(X_train, y_train)
  y_pred = clf.predict(X_test)
  print(clf.__class__.__name__, accuracy_score(y_test, y_pred))

LogisticRegression 0.875
RandomForestClassifier 1.0
SVC 0.995
VotingClassifier 0.995


##7.2 배깅과 페이스팅

*배깅은 하나의 예측기에 같은 샘플을 중복 할당하는 것을 허용한다.


*페이스팅은 샘플링할 때 중복을 허용하지 않는다.

###7.2.1 사이킷런의 배깅과 페이스팅

In [9]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_clf = BaggingClassifier(
    DecisionTreeClassifier(), n_estimators=500, max_samples=100, bootstrap=True, n_jobs=-1
)
bag_clf.fit(X_train, y_train)
y_pred = bag_clf.predict(X_test)

In [10]:
print(bag_clf.__class__.__name__, accuracy_score(y_test, y_pred))

BaggingClassifier 0.99


###7.2.2 oob 평가

In [11]:
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(), n_estimators = 500, bootstrap= True, n_jobs=-1, oob_score=True
)
bag_clf.fit(X_train, y_train)
bag_clf.oob_score_

0.9825

In [12]:
from sklearn.metrics import accuracy_score

y_pred = bag_clf.predict(X_test)
accuracy_score(y_test, y_pred)

0.985

In [13]:
bag_clf.oob_decision_function_

array([[0.        , 1.        ],
       [0.        , 1.        ],
       [0.        , 1.        ],
       ...,
       [0.80606061, 0.19393939],
       [0.        , 1.        ],
       [1.        , 0.        ]])

##7.3 랜덤 패치와 랜덤 서브스페이즈

랜덤 패치: 훈련 특성과 인스턴스를 모두 샘플링하는 것



-> 고차원의 데이터셋을 다룰 때 유용함(이미지 등) : 독립성 확보


랜덤 서브스페이스 방식: 훈련 샘플을 모두 사용하고 특성은 샘플링하는 방식

##7.4 랜덤 포레스트

In [15]:
from random import Random
from sklearn.ensemble import RandomForestClassifier

rnd_clf = RandomForestClassifier(n_estimators=500, max_leaf_nodes=16, n_jobs=-1)
rnd_clf.fit(X_train, y_train)

y_pred_rf = rnd_clf.predict(X_test)
accuracy_score(y_test, y_pred_rf)

0.995

In [19]:
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(max_leaf_nodes=16),
    n_estimators=500, max_samples=1.0, bootstrap=True, n_jobs=-1
)
bag_clf.fit(X_train, y_train)

y_pred_bag = bag_clf.predict(X_test)
accuracy_score(y_test, y_pred_bag)

1.0

##7.4.1 엑스트라 트리

익스트림 랜덤 트리(엑스트라 트리): 극단적으로 무작위한 트리의 랜덤 포레스트


트리를 더 무작위하게 만들기 위해 최적의 임곗값을 찾는 대신 후보 특성을 사용해 무작위로 분할한 다음 그중에서 최상의 분할을 선택하는 방식 사용

-> 편향이 증가하고 분산이 감소함



-> 일반적인 랜덤 포레스트보다 훨씬 빠름


*사이킷런의 ExtraTreesClassifier 사용

##7.4.2 특성 중요도

In [22]:
from sklearn.datasets import load_iris
iris = load_iris()
rnd_clf = RandomForestClassifier(n_estimators=500, n_jobs=-1)
rnd_clf.fit(iris['data'], iris['target'])
for name, score in zip(iris['feature_names'], rnd_clf.feature_importances_):
  print(name, score)

sepal length (cm) 0.10455139723101715
sepal width (cm) 0.025947964416395318
petal length (cm) 0.4580894633759822
petal width (cm) 0.4114111749766053
